# Probability Concepts with Real-World Examples

## Real-World Applications of Probability

### Autocorrect Systems
- **How it works**:
  - When you type "recieve", it suggests "receive"
  - Uses word frequency statistics (unigram/bigram probabilities)


### Spam Filters
- **How it works**:
  - Flags messages like "Win 1M!" as spam
  - Uses Bayesian probability to update beliefs


## Probability Terminology Explained

### Key Concepts with Email Examples

#### Random Experiment
- **Definition**: A process that produces an outcome
- **Email example**: Classifying an email as spam/ham
- **Other examples**: Rolling a die, flipping a coin

#### Trial
- **Definition**: One execution of a random experiment
- **Email example**: Processing one email message
- **Other examples**: One coin flip, one die roll

#### Outcome
- **Definition**: The result of a trial
- **Email example**: "spam" or "not spam" classification
- **Other examples**: "heads", "tails", die face number

#### Sample Space
- **Definition**: All possible outcomes
- **Email example**: All possible email texts and classifications
- **Notation**: S = {"spam", "not spam"}


### Corpus in Natural Language Processing

#### Definition
A **corpus** (plural: corpora) is a large, structured collection of texts used for linguistic analysis and training language models.

# Exercise 1: Next-Word Prediction



## Build Your Own Predictor

### Challenge
Design an algorithm that predicts the next word given typing history.

**Example Input/Output:**
- Input: "I want to"
- Output:
  - "go" (45% probability)
  - "eat" (30% probability)
  - "sleep" (25% probability)

## Next-Word Prediction Pipeline

### Step-by-Step Process
1. **Analyze current phrase**  
   Example: "How are"

2. **Generate likely candidates**  
   Potential next words: "you", "doing", "we"

3. **Rank by conditional probability**  
   Calculate P(word | context) for each candidate


# Conceptual implementation
context = "How are"
candidates = ["you", "doing", "we"]

# Calculate probabilities

Probabilities

    "you": 0.72,    # P("you" | "How are")

    "doing": 0.18,   # P("doing" | "How are")

    "we": 0.10       # P("we" | "How are")


# N-gram Language Models

## Markov Assumption
The probability of a word depends only on the previous **k** words:

P(w_n | w_1...w_{n-1}) ≈ P(w_n | w_{n-k}...w_{n-1})


## Unigram Model
**Assumption**: Words are completely independent  
**Probability**:
P(w) = count(w) / total_words

**Example**: "I love you"

P("I love you") = P("I") × P("love") × P("you")


## Bigram Model
**Assumption**: Word depends on 1 previous word  
**Probability**:
P(w_n | w_{n-1}) = count(w_{n-1} w_n) / count(w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"love")

## Trigram Model
**Assumption**: Word depends on 2 previous words  
**Probability**:
P(w_n | w_{n-2}, w_{n-1}) = count(w_{n-2} w_{n-1} w_n) / count(w_{n-2} w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"I love")

## Comparison Table

| Model    | Dependency          | Example Probability            |
|----------|---------------------|--------------------------------|
| Unigram  | None                | P("love")                     |
| Bigram   | 1 previous word     | P("you"/"love")              |
| Trigram  | 2 previous words    | P("you"/"I love")            |





### Dataset
Using Shakespeare's sonnets (public domain):
```python
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [1]:
!curl -o input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 63 1089k   63  690k    0     0  1140k      0 --:--:-- --:--:-- --:--:-- 1144k
100 1089k  100 1089k    0     0  1339k      0 --:--:-- --:--:-- --:--:-- 1343k


#Data Loading & Preprocessing

In [2]:
def load_data(filepath):
    """Load text file and return as string"""
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

def preprocess(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters
    3. Splitting into tokens (words)
    """
    import re

    # Convert to lowercase
    text = text.lower()


    text = re.sub(r"[^a-z0-9'\s]", "", text)

    # Split into tokens (words)
    tokens = text.split()

    return tokens

# TEST
text = load_data("input.txt")
tokens = preprocess(text)
print("First 50 tokens:", tokens[:50])

First 50 tokens: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen', 'first', 'you', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', 'all', 'we', "know't", 'we', "know't", 'first', 'citizen', 'let', 'us']


# Model Implementation

In [3]:
def build_ngram_model(tokens, n=2):
    """
    Build n-gram model from tokens.
    Currently optimized for n=2 (Bigram).
    """
    model = {}

    for i in range(len(tokens) - (n - 1)):
        context = tokens[i]
        target = tokens[i + 1]

        if context not in model:
            model[context] = {}

        if target not in model[context]:
            model[context][target] = 0
            
        model[context][target] += 1

    return model

def predict_next_word(model, context, k=3):
    """
    Predict next words given context (a single word for bigrams).
    Calculates Conditional Probability: P(word | context)
    """
    context = context.lower()
    
    if context not in model:
        return []

    candidates = model[context]
    total_occurrences = sum(candidates.values())

    predictions = []
    for word, count in candidates.items():
        probability = count / total_occurrences
        predictions.append((word, probability))

    predictions.sort(key=lambda x: x[1], reverse=True)
    
    return predictions[:k]

shakespeare_model = build_ngram_model(tokens, n=2)

print("Model built and ready for testing!")

Model built and ready for testing!


#Testing

In [4]:
bigram_model = build_ngram_model(tokens, n=2)
print("Words after 'the':", predict_next_word(bigram_model, "the"))

Words after 'the': [('king', 0.024685459468068164), ('duke', 0.0168816690555821), ('world', 0.015766841853798376)]


# Exercise 2: Spam/Ham Classification



## Spam vs Ham Examples

<div style="display: flex;">
<div style="flex: 50%; padding: 10px;">

### Spam Examples
- "Claim your free \$1M"
- "You won an iPhone!"
- "Limited time offer!"
- "Click here to claim your prize"

</div>
<div style="flex: 50%; padding: 10px;">

### Ham Examples
- "Meeting at 3 PM"
- "Project update attached"
- "Let's discuss the proposal"
- "Quarterly report is ready"

</div>
</div>


### 1. Prior Probability (P(c))
**What it represents:**  
The baseline probability of a class before examining the message content.

**Example Calculation:**  

#### In 100 emails:
spam_count = 30

ham_count = 70

P_spam = spam_count/100  # 0.30

P_ham = ham_count/100    # 0.70

### 2. Likelihood (P(w|c))

**Definition:**  
The probability of observing a specific word given a particular class (spam/ham). This measures how characteristic a word is for a certain class.


#### Spam class: 25 occurrences out of 1000 total words
P_free_spam = 25/1000  # 0.025

#### Ham class: 2 occurrences out of 3000 total words
P_free_ham = 2/3000    # ≈0.00067

### 3. Posterior Probability

**Definition:**
The posterior probability **P(c|msg)** represents:
- The probability that a given message belongs to class *c* (spam/ham)
- The key quantity we compute to make classification decisions

### Bayesian Classification

### Decision Rule
The classifier selects the class with the highest posterior probability:

Where:
- `P(c|msg)` = Posterior probability (what we want to calculate)
- `P(msg|c)` = Likelihood of the message given the class
- `P(c)` = Prior probability of the class

### Practical Example: "free prize" message

**Given Probabilities:**

### Word probabilities
P_free_spam = 0.025

P_free_ham = 0.00067

P_prize_spam = 0.015

P_prize_ham = 0.001



### Class priors
P_spam = 0.3

P_ham = 0.7

### Calculation Steps
#### Compute Spam Probability:
P_msg_spam = P_free_spam * P_prize_spam * P_spam
           = 0.025 × 0.015 × 0.3
           ≈ 0.0001125

####Compute Ham Probability:

P_msg_ham = P_free_ham * P_prize_ham * P_ham
          = 0.00067 × 0.001 × 0.7
          ≈ 0.000000469

####Comparison:
0.0001125 (Spam) > 0.000000469 (Ham)

##Laplace Smoothing
**Problem**:  
P("xyz"|Spam) = 0 if "xyz" never appeared in training spam

**Solution**:  
$$ P_{\text{smooth}}(w|c) = \frac{\text{count}(w,c) + 1}{\text{count}(c) + V} $$

*Example*:  
- V = 10,000 words  
- count("prize", Spam) = 0  
- count(Spam) = 1000  
$$ P_{\text{smooth}}("prize"|Spam) = \frac{0+1}{1000+10000} \approx 0.00009 $$



## Stop Words in Text Processing

**Definition:** Common words (e.g., "the", "a", "and") that appear frequently but carry minimal meaning.

**Why Remove Them?**
1. **Meaning Preservation:**  
   "The quick brown fox" → "quick brown fox" (keeps core meaning)
   
2. **Improved Analysis:**  
   - Without: P("the"|spam)=0.101 vs P("the"|ham)=0.098 (useless)  
   - With: P("free"|spam)=0.025 vs P("free"|ham)=0.00067 (discriminative)

3. **Efficiency:**  
   Reduces vocabulary size by 30-40% (1000 words → 600 words)

**Example: Spam Detection**
```python
from sklearn.feature_extraction.text import CountVectorizer

### With stop words
text = ["You have won a free prize"]
vectorizer = CountVectorizer()
print("With stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1 1 1]] (for "you", "have", "won", "free", "prize")

### Without stop words
vectorizer = CountVectorizer(stop_words='english')
print("Without stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1]] (for "won", "free", "prize")

## Dataset Loading

In [5]:
import requests
import zipfile
import io

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
print("Downloading dataset...")
response = requests.get(url)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall()
    print("Files extracted successfully:", z.namelist())

Files extracted successfully: ['SMSSpamCollection', 'readme']


In [6]:
def load_sms_data(filepath):
    """Load SMS spam data from file.
    Returns: (list of messages, list of labels)"""
    messages = []
    labels = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labels.append(parts[0])
                messages.append(parts[1])
    return messages, labels

def preprocess_text(text, remove_stopwords=True):
    """Clean text with optional stopword removal"""
    import re

    # Basic cleaning
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        stop_words = {
            'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves',
            'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him',
            'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its',
            'itself', 'they', 'them', 'their', 'theirs', 'themselves',
            'what', 'which', 'who', 'whom', 'this', 'that', 'these',
            'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
            'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
            'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as',
            'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
            'against', 'between', 'into', 'through', 'during', 'before',
            'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in',
            'out', 'on', 'off', 'over', 'under', 'again', 'further',
            'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
            'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other',
            'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same',
            'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just',
            'don', 'should', 'now'
        }

        words = text.split()
        filtered_words = [word for word in words if word not in stop_words]
        text = ' '.join(filtered_words)

    return text

# Test with stopword removal
messages, labels = load_sms_data("SMSSpamCollection")
print("Original:", messages[0])
print("Processed:", preprocess_text(messages[0]))
print("\nWithout stopwords:", preprocess_text(messages[0], remove_stopwords=True))

Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: go jurong point crazy available bugis n great world la e buffet cine got amore wat

Without stopwords: go jurong point crazy available bugis n great world la e buffet cine got amore wat


#Calculate Prior Probabilities

In [7]:
def calculate_priors(labels):

    total = len(labels)
    
    spam_count = labels.count('spam')
    ham_count = labels.count('ham')
    
    return {
        'spam': spam_count / total,
        'ham': ham_count / total
    }

# Test Priors
priors = calculate_priors(labels)
print(f"P(Spam): {priors['spam']:.2%}")

P(Spam): 13.40%


#Count Words per Class

In [8]:
from collections import Counter

def count_words(messages, labels):
    spam_words = []
    ham_words = []
    vocab = set()

    for msg, label in zip(messages, labels):

        words = preprocess_text(msg).split()
 
        if label == 'spam':
            spam_words.extend(words)
        else:
            ham_words.extend(words)
            
        vocab.update(words)

    spam_counts = Counter(spam_words)
    ham_counts = Counter(ham_words)

    return spam_counts, ham_counts, vocab

# TEST
spam_counts, ham_counts, vocab = count_words(messages, labels)

print(f"Total Unique Words: {len(vocab)}")
print(f"Free in Spam: {spam_counts['free']}")
print(f"Free in Ham: {ham_counts['free']}")

Total Unique Words: 9464
Free in Spam: 216
Free in Ham: 59


#Compute Word Probabilities (with Smoothing)

In [9]:
def calculate_word_probs(spam_counts, ham_counts, vocab, k=1):
    word_probs = {'spam': {}, 'ham': {}}
    
    total_spam = sum(spam_counts.values())
    total_ham = sum(ham_counts.values())
    
    v_size = len(vocab)
    
    for word in vocab:
        
        word_probs['spam'][word] = (spam_counts[word] + k) / (total_spam + k * v_size)
        
        word_probs['ham'][word] = (ham_counts[word] + k) / (total_ham + k * v_size)
        
    return word_probs

# --- Test ---
probs = calculate_word_probs(spam_counts, ham_counts, vocab)

print("✅ Success! Probabilities calculated.")
print(f"Probability of 'free' in Spam: {probs['spam'].get('free', 0):.6f}")
print(f"Probability of 'free' in Ham:  {probs['ham'].get('free', 0):.6f}")

✅ Success! Probabilities calculated.
Probability of 'free' in Spam: 0.009818
Probability of 'free' in Ham:  0.001205


#Predict Spam Probability

In [10]:
import math
from collections import defaultdict

def calculate_priors(labels):
    total = len(labels)
    prior_prob = {
        'spam': labels.count('spam') / total,
        'ham': labels.count('ham') / total
    }
    return prior_prob

def count_words(messages, labels):
    word_counts = {'spam': defaultdict(int), 'ham': {}}
    word_counts['ham'] = defaultdict(int)
    vocab = set()

    for msg, label in zip(messages, labels):
        words = preprocess_text(msg).split()
        for word in words:
            word_counts[label][word] += 1
            vocab.add(word)
    return word_counts, vocab

def calculate_word_probs(word_counts, vocab, k=1):
    word_probs = {'spam': {}, 'ham': {}}
    v = len(vocab)
    
    total_spam_words = sum(word_counts['spam'].values())
    total_ham_words = sum(word_counts['ham'].values())

    for word in vocab:
        word_probs['spam'][word] = (word_counts['spam'][word] + k) / (total_spam_words + k * v)
        word_probs['ham'][word] = (word_counts['ham'][word] + k) / (total_ham_words + k * v)
        
    return word_probs

def predict(message, prior_prob, word_probs, vocab):
    words = preprocess_text(message).split()
    
    log_prob_spam = math.log(prior_prob['spam'])
    log_prob_ham = math.log(prior_prob['ham'])
    
    for word in words:
        if word in vocab:
            log_prob_spam += math.log(word_probs['spam'][word])
            log_prob_ham += math.log(word_probs['ham'][word])
            
    spam_prob = math.exp(log_prob_spam)
    ham_prob = math.exp(log_prob_ham)
    
    probability = spam_prob / (spam_prob + ham_prob)
    return probability

In [11]:
# Test calculate_priors()
labels = ['spam', 'ham', 'ham']
print("Priors:", calculate_priors(labels))

# Test count_words()
messages = ["free prize", "meeting today"]
test_labels = ['spam', 'ham']
word_counts, vocab = count_words(messages, test_labels)
print("Count 'free' in Spam:", word_counts['spam']['free']) 
print("Vocab Size:", len(vocab))

# Test calculate_word_probs()
word_probs = calculate_word_probs(word_counts, vocab, k=1)
print("Prob 'free' in Spam:", round(word_probs['spam']['free'], 2))

# Final Predict Test
test_msg = "free prize"
print(f"Is '{test_msg}' spam? Chance: {predict(test_msg, calculate_priors(labels), word_probs, vocab):.2f}")

Priors: {'spam': 0.3333333333333333, 'ham': 0.6666666666666666}
Count 'free' in Spam: 1
Vocab Size: 4
Prob 'free' in Spam: 0.33
Is 'free prize' spam? Chance: 0.67
